# 🌸 Iris Flower Classification — Enhanced Notebook

**Updated Features:**
- ✅ Original ML Classification (Logistic Regression)
- ✅ Multiple Model Comparison (LR, RF, SVM)
- 🆕 Computer Vision Analysis
- 🆕 AI Chatbot Logic
- 🆕 Advanced Visualizations

## Step 1: Import Required Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# ML Libraries
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.decomposition import PCA

# Computer Vision Libraries
from PIL import Image, ImageDraw, ImageFilter, ImageEnhance
import cv2
import io
import base64

print('✅ All libraries imported successfully!')

## Step 2: Load & Explore Dataset

In [ ]:
# Load from CSV
try:
    df = pd.read_csv('data/Iris.csv')
    df.columns = [c.replace('Cm','').strip() for c in df.columns]
    if 'Id' in df.columns:
        df = df.drop('Id', axis=1)
    df['Species'] = df['Species'].str.replace('Iris-', '')
    print('✅ Loaded from CSV')
except:
    iris = load_iris()
    df = pd.DataFrame(iris.data, columns=['SepalLength','SepalWidth','PetalLength','PetalWidth'])
    df['Species'] = [iris.target_names[t] for t in iris.target]
    print('✅ Loaded from sklearn')

print(f'Shape: {df.shape}')
df.head()

In [ ]:
# Dataset Info
print('=== Dataset Info ===')
df.info()
print('\n=== Statistics ===')
df.describe()

In [ ]:
print('=== Class Distribution ===')
print(df['Species'].value_counts())
print('\n=== Missing Values ===')
print(df.isnull().sum())

## Step 3: Advanced Data Visualization

In [ ]:
COLORS = {'setosa': '#11998e', 'versicolor': '#2196F3', 'virginica': '#f5576c'}
FEATURES = ['SepalLength', 'SepalWidth', 'PetalLength', 'PetalWidth']

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Iris Dataset — Feature Distributions', fontsize=16, fontweight='bold')

for idx, feat in enumerate(FEATURES):
    ax = axes[idx//2][idx%2]
    for species, color in COLORS.items():
        data = df[df['Species']==species][feat]
        ax.hist(data, bins=15, alpha=0.6, color=color, label=species, edgecolor='white')
    ax.set_title(feat, fontweight='bold')
    ax.set_xlabel('Value (cm)')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('assets/feature_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Pairplot
sns.set_style('whitegrid')
palette = COLORS
g = sns.pairplot(df, hue='Species', palette=palette, plot_kws={'alpha': 0.6})
g.fig.suptitle('Pairplot — All Features', y=1.02, fontsize=14, fontweight='bold')
plt.savefig('assets/pairplot.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Correlation Heatmap
fig, ax = plt.subplots(figsize=(8, 6))
corr = df[FEATURES].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', 
            ax=ax, square=True, linewidths=0.5, cbar_kws={'shrink':0.8})
ax.set_title('Feature Correlation Matrix', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.savefig('assets/correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 4: PCA — Dimensionality Reduction

In [ ]:
X = df[FEATURES].values
y = df['Species'].values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# PCA 2D
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 2D PCA scatter
for species, color in COLORS.items():
    mask = y == species
    axes[0].scatter(X_pca[mask, 0], X_pca[mask, 1], 
                   c=color, label=species, alpha=0.7, s=60, edgecolors='white')
axes[0].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)')
axes[0].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)')
axes[0].set_title('PCA 2D Projection', fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Explained variance
pca_full = PCA()
pca_full.fit(X_scaled)
cumvar = np.cumsum(pca_full.explained_variance_ratio_)
axes[1].bar(range(1, len(cumvar)+1), pca_full.explained_variance_ratio_, 
           color='#667eea', alpha=0.7, label='Individual')
axes[1].plot(range(1, len(cumvar)+1), cumvar, 'o-', color='#764ba2', label='Cumulative')
axes[1].axhline(y=0.95, color='red', linestyle='--', alpha=0.5, label='95% threshold')
axes[1].set_xlabel('Number of Components')
axes[1].set_ylabel('Explained Variance Ratio')
axes[1].set_title('PCA Explained Variance', fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('assets/pca_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'✅ 2 components explain {sum(pca.explained_variance_ratio_):.1%} of variance')

## Step 5: Multiple ML Models Training & Comparison

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

models = {
    'Logistic Regression': LogisticRegression(max_iter=500),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'SVM': SVC(probability=True, random_state=42),
    'KNN': KNeighborsClassifier(n_neighbors=5),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42)
}

results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    cv_scores = cross_val_score(model, X_scaled, y, cv=5)
    results[name] = {
        'model': model,
        'test_accuracy': accuracy_score(y_test, y_pred),
        'cv_mean': cv_scores.mean(),
        'cv_std': cv_scores.std(),
        'predictions': y_pred
    }
    print(f'{name}: Test={results[name]["test_accuracy"]:.3f}, CV={cv_scores.mean():.3f}±{cv_scores.std():.3f}')

best_name = max(results, key=lambda x: results[x]['test_accuracy'])
print(f'\n🏆 Best Model: {best_name} ({results[best_name]["test_accuracy"]:.1%} accuracy)')

In [ ]:
# Model comparison chart
names = list(results.keys())
test_accs = [results[n]['test_accuracy'] for n in names]
cv_means = [results[n]['cv_mean'] for n in names]
cv_stds = [results[n]['cv_std'] for n in names]

x = np.arange(len(names))
width = 0.35

fig, ax = plt.subplots(figsize=(12, 6))
bars1 = ax.bar(x - width/2, test_accs, width, label='Test Accuracy', color='#667eea', alpha=0.8)
bars2 = ax.bar(x + width/2, cv_means, width, label='CV Mean', color='#764ba2', alpha=0.8,
               yerr=cv_stds, capsize=5)

ax.set_ylim(0.85, 1.02)
ax.set_xlabel('Model', fontsize=12)
ax.set_ylabel('Accuracy', fontsize=12)
ax.set_title('Model Comparison — Test vs Cross-Validation Accuracy', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(names, rotation=15, ha='right')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

for bar in bars1:
    ax.annotate(f'{bar.get_height():.3f}', xy=(bar.get_x() + bar.get_width()/2, bar.get_height()),
                xytext=(0, 3), textcoords='offset points', ha='center', fontsize=9)

plt.tight_layout()
plt.savefig('assets/model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Best model detailed report
best_model = results[best_name]['model']
y_pred_best = results[best_name]['predictions']

print(f'=== {best_name} — Classification Report ===')
print(classification_report(y_test, y_pred_best))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred_best)
species_list = np.unique(y)

fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Purples',
            xticklabels=species_list, yticklabels=species_list,
            linewidths=0.5, ax=ax)
ax.set_xlabel('Predicted', fontsize=12)
ax.set_ylabel('Actual', fontsize=12)
ax.set_title(f'{best_name} — Confusion Matrix', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('assets/confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Random Forest Feature Importance
rf = results['Random Forest']['model']
importances = rf.feature_importances_
indices = np.argsort(importances)[::-1]

fig, ax = plt.subplots(figsize=(8, 5))
colors = ['#667eea', '#764ba2', '#11998e', '#f5576c']
bars = ax.bar(FEATURES, importances, color=colors, alpha=0.8, edgecolor='white')
ax.set_title('Random Forest — Feature Importance', fontsize=13, fontweight='bold')
ax.set_xlabel('Feature')
ax.set_ylabel('Importance Score')
ax.grid(True, alpha=0.3, axis='y')

for bar, val in zip(bars, importances):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{val:.3f}', ha='center', fontsize=10, fontweight='500')

plt.tight_layout()
plt.savefig('assets/feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

## 🆕 Step 6: Computer Vision Analysis

Using PIL and OpenCV to analyze synthetic Iris flower images — demonstrating how CV pipelines work for flower classification.

In [ ]:
# ── CV Function: Create synthetic flower image for demo ──
def create_iris_image(species='setosa', size=(400, 400)):
    """Creates a synthetic Iris flower image using PIL."""
    # Background
    img = Image.new('RGB', size, color=(240, 248, 240))
    draw = ImageDraw.Draw(img)
    
    cx, cy = size[0]//2, size[1]//2
    
    # Species-specific parameters
    params = {
        'setosa':     {'petal_r': 40, 'sepal_r': 75, 'color': (100, 220, 150), 'sepal': (60, 180, 100)},
        'versicolor': {'petal_r': 65, 'sepal_r': 95, 'color': (100, 160, 230), 'sepal': (60, 110, 190)},
        'virginica':  {'petal_r': 85, 'sepal_r': 115,'color': (210, 100, 190), 'sepal': (170, 60, 150)}
    }
    p = params.get(species, params['setosa'])
    
    # Draw stem
    draw.line([(cx, cy+p['sepal_r']+10), (cx, cy+p['sepal_r']+80)],
              fill=(60, 140, 60), width=6)
    
    # Draw sepals (3, rotated 120°)
    for angle in [0, 120, 240]:
        rad = np.radians(angle)
        x1 = cx + int(p['sepal_r'] * np.cos(rad))
        y1 = cy + int(p['sepal_r'] * np.sin(rad))
        draw.ellipse([x1-20, y1-12, x1+20, y1+12], fill=p['sepal'], outline=(255,255,255), width=2)
    
    # Draw petals (3, rotated 120°, offset 60°)
    for angle in [60, 180, 300]:
        rad = np.radians(angle)
        x1 = cx + int(p['petal_r'] * np.cos(rad))
        y1 = cy + int(p['petal_r'] * np.sin(rad))
        draw.ellipse([x1-15, y1-10, x1+15, y1+10], fill=p['color'], outline=(255,255,255), width=2)
    
    # Center
    draw.ellipse([cx-18, cy-18, cx+18, cy+18], fill=(255, 220, 80), outline=(200,160,30), width=2)
    draw.ellipse([cx-8, cy-8, cx+8, cy+8], fill=(255, 180, 20))
    
    # Soft glow
    img = img.filter(ImageFilter.GaussianBlur(radius=0.5))
    return img

# Generate all 3
fig, axes = plt.subplots(1, 3, figsize=(12, 5))
fig.suptitle('Synthetic Iris Flower Images (Computer Vision Demo)', fontsize=14, fontweight='bold')

for ax, species in zip(axes, ['setosa', 'versicolor', 'virginica']):
    img = create_iris_image(species)
    ax.imshow(np.array(img))
    ax.set_title(f'Iris {species.capitalize()}', fontweight='bold', fontsize=12)
    ax.axis('off')

plt.tight_layout()
plt.savefig('assets/synthetic_flowers.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Synthetic flower images generated!')

In [ ]:
# ── CV Analysis: Extract visual features from synthetic images ──
def extract_cv_features(img_pil):
    """Extract visual features from a PIL image."""
    img_arr = np.array(img_pil)
    
    # Convert to OpenCV format
    img_cv = cv2.cvtColor(img_arr, cv2.COLOR_RGB2BGR)
    gray = cv2.cvtColor(img_cv, cv2.COLOR_BGR2GRAY)
    
    # Color stats
    r_mean = img_arr[:,:,0].mean()
    g_mean = img_arr[:,:,1].mean()
    b_mean = img_arr[:,:,2].mean()
    
    # Brightness
    brightness = gray.mean()
    contrast = gray.std()
    
    # Edge detection using Canny
    edges = cv2.Canny(gray, 50, 150)
    edge_density = edges.mean()
    
    # Texture (Laplacian variance)
    texture = cv2.Laplacian(gray, cv2.CV_64F).var()
    
    # HSV color space
    hsv = cv2.cvtColor(img_cv, cv2.COLOR_BGR2HSV)
    hue_mean = hsv[:,:,0].mean()
    sat_mean = hsv[:,:,1].mean()
    
    return {
        'R_Mean': round(r_mean, 2), 'G_Mean': round(g_mean, 2), 'B_Mean': round(b_mean, 2),
        'Brightness': round(brightness, 2), 'Contrast': round(contrast, 2),
        'Edge_Density': round(edge_density, 4), 'Texture': round(texture, 2),
        'Hue': round(hue_mean, 2), 'Saturation': round(sat_mean, 2)
    }

# Extract features for all 3 species
cv_features = []
for species in ['setosa', 'versicolor', 'virginica']:
    img = create_iris_image(species)
    feats = extract_cv_features(img)
    feats['Species'] = species
    cv_features.append(feats)

cv_df = pd.DataFrame(cv_features)
print('=== Computer Vision Features ===')
cv_df

In [ ]:
# Visualize CV features
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
fig.suptitle('Computer Vision Feature Analysis', fontsize=14, fontweight='bold')

cv_feature_names = ['R_Mean', 'G_Mean', 'B_Mean', 'Brightness', 'Edge_Density', 'Texture']
species_list = cv_df['Species'].tolist()
color_list = [COLORS[s] for s in species_list]

for ax, feat in zip(axes.flatten(), cv_feature_names):
    values = cv_df[feat].values
    bars = ax.bar(species_list, values, color=color_list, alpha=0.8, edgecolor='white')
    ax.set_title(feat, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='y')
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height(),
                f'{val:.1f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig('assets/cv_features.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ CV Feature analysis complete!')

In [ ]:
# ── Edge Detection Demo ──
fig, axes = plt.subplots(3, 4, figsize=(14, 10))
fig.suptitle('Computer Vision Processing Pipeline', fontsize=14, fontweight='bold')

for row, species in enumerate(['setosa', 'versicolor', 'virginica']):
    img_pil = create_iris_image(species)
    img_arr = np.array(img_pil)
    img_cv = cv2.cvtColor(img_arr, cv2.COLOR_RGB2BGR)
    gray = cv2.cvtColor(img_cv, cv2.COLOR_BGR2GRAY)
    edges = cv2.Canny(gray, 50, 150)
    blurred = cv2.GaussianBlur(gray, (11,11), 0)
    
    # Original
    axes[row][0].imshow(img_arr)
    axes[row][0].set_title(f'{species.capitalize()} — Original')
    
    # Grayscale
    axes[row][1].imshow(gray, cmap='gray')
    axes[row][1].set_title('Grayscale')
    
    # Edge Detection
    axes[row][2].imshow(edges, cmap='plasma')
    axes[row][2].set_title('Edge Detection (Canny)')
    
    # RGB Histogram
    for ch, color in enumerate(['r','g','b']):
        hist = cv2.calcHist([img_cv], [ch], None, [64], [0, 256])
        axes[row][3].plot(hist, color=color, alpha=0.7)
    axes[row][3].set_title('RGB Histogram')
    axes[row][3].set_xlim(0, 64)

for ax in axes.flatten():
    ax.axis('off') if ax.get_images() else ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('assets/cv_pipeline.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ CV Pipeline visualization complete!')

In [ ]:
# ── Dominant Color Extraction using K-Means ──
def get_dominant_colors(img_pil, n_colors=5):
    from sklearn.cluster import KMeans
    img_small = np.array(img_pil.resize((80, 80))).reshape(-1, 3).astype(float)
    km = KMeans(n_clusters=n_colors, random_state=42, n_init=10)
    km.fit(img_small)
    centers = km.cluster_centers_.astype(int)
    counts = np.bincount(km.labels_)
    sorted_idx = np.argsort(-counts)
    return centers[sorted_idx], counts[sorted_idx]

fig, axes = plt.subplots(3, 2, figsize=(10, 10))
fig.suptitle('Dominant Color Extraction (K-Means)', fontsize=14, fontweight='bold')

for row, species in enumerate(['setosa', 'versicolor', 'virginica']):
    img = create_iris_image(species)
    colors, counts = get_dominant_colors(img)
    total = counts.sum()
    
    axes[row][0].imshow(np.array(img))
    axes[row][0].set_title(f'{species.capitalize()}')
    axes[row][0].axis('off')
    
    # Color swatches
    for i, (c, cnt) in enumerate(zip(colors, counts)):
        axes[row][1].barh(i, cnt/total*100, 
                         color=c/255, height=0.7)
        axes[row][1].text(cnt/total*100 + 0.5, i,
                         f'#{c[0]:02x}{c[1]:02x}{c[2]:02x} ({cnt/total*100:.1f}%)',
                         va='center', fontsize=9)
    axes[row][1].set_xlim(0, 45)
    axes[row][1].set_title('Dominant Colors')
    axes[row][1].axis('off')

plt.tight_layout()
plt.savefig('assets/dominant_colors.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Dominant color extraction complete!')

## 🆕 Step 7: AI Chatbot Logic

Implementing a rule-based intelligent chatbot for Iris dataset queries.

In [ ]:
class IrisChatbot:
    """Iris Expert AI Chatbot — answers questions about the Iris dataset."""
    
    def __init__(self, df, models, scaler):
        self.df = df
        self.models = models
        self.scaler = scaler
        self.features = ['SepalLength', 'SepalWidth', 'PetalLength', 'PetalWidth']
        self.colors = {'setosa':'🟢', 'versicolor':'🔵', 'virginica':'🟣'}
        print('🌸 IrisChatbot initialized! Ready to answer questions.')
    
    def respond(self, user_input):
        import re
        msg = user_input.lower().strip()
        
        # Greetings
        if any(w in msg for w in ['hello', 'hi', 'hey', 'namaste']):
            return '👋 Hello! I am Iris Expert AI. Ask me about species, predictions, or dataset stats!'
        
        # Prediction
        nums = re.findall(r'\d+\.?\d*', msg)
        if len(nums) >= 4 and any(w in msg for w in ['predict', 'classify', 'species', 'what is']):
            sl, sw, pl, pw = float(nums[0]), float(nums[1]), float(nums[2]), float(nums[3])
            sample = self.scaler.transform([[sl, sw, pl, pw]])
            pred = self.models['Random Forest'].predict(sample)[0]
            proba = self.models['Random Forest'].predict_proba(sample)[0]
            classes = self.models['Random Forest'].classes_
            best_conf = max(proba)
            icon = self.colors.get(pred, '🌸')
            return (f'🔮 Prediction: {icon} Iris {pred.capitalize()} ({best_conf:.1%} confidence)\n'
                    f'Probabilities: ' + ', '.join([f'{c}: {p:.1%}' for c, p in zip(classes, proba)]))
        
        # Species info
        for species in ['setosa', 'versicolor', 'virginica']:
            if species in msg:
                stats = self.df[self.df['Species']==species][self.features].describe().loc['mean']
                icon = self.colors[species]
                return (f'{icon} Iris {species.capitalize()} Average Measurements:\n'
                        + '\n'.join([f'  • {f}: {v:.2f} cm' for f, v in stats.items()]))
        
        # Model info
        if any(w in msg for w in ['accuracy', 'model', 'best', 'performance']):
            res_str = '\n'.join([f'  • {name}: {r["test_accuracy"]:.1%}' 
                                for name, r in results.items()])
            best = max(results, key=lambda x: results[x]['test_accuracy'])
            return f'🤖 Model Accuracies:\n{res_str}\n\n🏆 Best: {best} ({results[best]["test_accuracy"]:.1%})'
        
        # Dataset stats
        if any(w in msg for w in ['dataset', 'data', 'size', 'how many', 'samples']):
            return (f'📊 Dataset: {len(self.df)} samples, {len(self.features)} features, '
                    f'3 species (50 each)\nFeatures: {', '.join(self.features)}')
        
        # Feature stats
        for feat in self.features:
            if feat.lower() in msg:
                stats = self.df[feat].describe()
                return (f'📈 {feat} Statistics:\n'
                        f'  Mean: {stats["mean"]:.3f} | Std: {stats["std"]:.3f}\n'
                        f'  Min: {stats["min"]:.3f} | Max: {stats["max"]:.3f}')
        
        return '🤔 I did not understand that. Try: species names, predict with measurements, or ask about models!'

# Initialize chatbot
chatbot = IrisChatbot(df, {n: r['model'] for n, r in results.items()}, scaler)

# Demo conversation
demo_queries = [
    "Hello!",
    "Tell me about setosa",
    "What is the best model accuracy?",
    "Predict species: 5.1, 3.5, 1.4, 0.2",
    "How many samples are in the dataset?",
    "Tell me about PetalLength"
]

print('=== AI Chatbot Demo Conversation ===')
print('='*50)
for q in demo_queries:
    print(f'\n👤 User: {q}')
    print(f'🌸 Bot:  {chatbot.respond(q)}')
    print('-'*40)

## Step 8: Final Prediction & Save Model

In [ ]:
import pickle
import os

os.makedirs('models', exist_ok=True)

# Save best model and scaler
best_model_obj = results[best_name]['model']
with open('models/iris_model.pkl', 'wb') as f:
    pickle.dump(best_model_obj, f)
with open('models/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

print(f'✅ Saved: {best_name} → models/iris_model.pkl')
print(f'✅ Saved: StandardScaler → models/scaler.pkl')

# Test prediction
test_samples = [
    ([5.1, 3.5, 1.4, 0.2], 'setosa'),
    ([6.3, 3.3, 4.7, 1.6], 'versicolor'),
    ([6.9, 3.1, 5.4, 2.1], 'virginica'),
]

print('\n=== Sample Predictions ===')
for sample, expected in test_samples:
    scaled = scaler.transform([sample])
    pred = best_model_obj.predict(scaled)[0]
    proba = best_model_obj.predict_proba(scaled)[0]
    conf = max(proba)
    status = '✅' if pred == expected else '❌'
    print(f'{status} Input: {sample} → Predicted: {pred} (conf: {conf:.1%}) | Expected: {expected}')

In [ ]:
# Export updated dataset
df.to_csv('data/iris_processed.csv', index=False)
print('✅ Saved: data/iris_processed.csv')
print('\n🎉 Notebook execution complete! All assets saved.')
print('\n📂 Generated Files:')
for f in ['assets/feature_distributions.png', 'assets/pairplot.png', 
          'assets/correlation_heatmap.png', 'assets/pca_analysis.png',
          'assets/model_comparison.png', 'assets/confusion_matrix.png',
          'assets/feature_importance.png', 'assets/synthetic_flowers.png',
          'assets/cv_features.png', 'assets/cv_pipeline.png', 
          'assets/dominant_colors.png', 'models/iris_model.pkl']:
    print(f'  📄 {f}')